# 04 — SHAP + OOD (T4)


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

try:
    import shap
    HAS_SHAP = True
except Exception as e:
    shap = None
    HAS_SHAP = False
    print(f"SHAP unavailable in current environment, fallback to permutation importance. Reason: {e}")

from src.dataset import OpenBanditDataset

sns.set_theme(style="whitegrid")
np.random.seed(42)

POLICY_RANDOM = "random"
POLICY_BTS = "bts"
CAMPAIGN = "all"
N_ACTIONS = 80
N_CONTEXT_FEATURES = 30
RANDOM_STATE = 42

FIGURES_DIR = Path("../figures/week4")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
dataset_random = OpenBanditDataset(behavior_policy=POLICY_RANDOM, campaign=CAMPAIGN)
feedback_random = dataset_random.obtain_batch_bandit_feedback()

dataset_bts = OpenBanditDataset(behavior_policy=POLICY_BTS, campaign=CAMPAIGN)
feedback_bts = dataset_bts.obtain_batch_bandit_feedback()

CONTEXT_DIM = feedback_random["context"].shape[1]

context_random = feedback_random["context"][:, :CONTEXT_DIM]
action_random = feedback_random["action"]
reward_random = feedback_random["reward"].astype(np.float32)

X_full = np.hstack([context_random, np.eye(N_ACTIONS)[action_random]])
y_full = reward_random

X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_full
)

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(n_neg / max(n_pos, 1)),
    eval_metric="logloss",
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)

print(f"Model fitted for diagnostics. CONTEXT_DIM={CONTEXT_DIM}")


In [ ]:
val_idx = np.random.choice(len(X_val), size=min(3000, len(X_val)), replace=False)
X_val_sample = X_val[val_idx]
y_val_sample = y_val[val_idx]

context_dim_model = int(model.n_features_in_ - N_ACTIONS)
feature_names = [f"ctx_{i}" for i in range(context_dim_model)] + [f"act_{i}" for i in range(N_ACTIONS)]

if HAS_SHAP:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_val_sample)
    mean_abs = np.abs(shap_values).mean(axis=0)
    importance_col = "mean_abs_importance"
    title_metric = "mean |SHAP|"
else:
    pfi = permutation_importance(
        model,
        X_val_sample,
        y_val_sample,
        n_repeats=5,
        random_state=RANDOM_STATE,
        scoring="average_precision",
    )
    mean_abs = np.abs(pfi.importances_mean)
    importance_col = "mean_abs_importance"
    title_metric = "mean |permutation importance|"

importance_df = (
    pd.DataFrame({"feature": feature_names, importance_col: mean_abs})
    .sort_values(importance_col, ascending=False)
    .reset_index(drop=True)
)

topk = 20
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=importance_df.head(topk), x=importance_col, y="feature", ax=ax, color="#4C78A8")
ax.set_title(f"Week 4: Top-{topk} features by {title_metric}")
ax.set_xlabel(title_metric)
ax.set_ylabel("feature")
plt.tight_layout()
out_path = FIGURES_DIR / "week4_shap_top20.png"
plt.savefig(out_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved figure: {out_path}")

importance_df.head(10)


In [ ]:
context_bts = feedback_bts["context"][:, :CONTEXT_DIM]

sample_train_idx = np.random.choice(len(context_random), size=min(40000, len(context_random)), replace=False)
random_context_sample = context_random[sample_train_idx]

sample_bts_idx = np.random.choice(len(context_bts), size=min(40000, len(context_bts)), replace=False)
bts_context_sample = context_bts[sample_bts_idx]

nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
nn.fit(random_context_sample)

distances, _ = nn.kneighbors(bts_context_sample)
distances = distances.squeeze()

ood_threshold = np.quantile(distances, 0.95)
ood_rate = float((distances > ood_threshold).mean())

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(distances, bins=80, color="#72B7B2", alpha=0.9)
ax.axvline(ood_threshold, color="crimson", linestyle="--", linewidth=2, label=f"95th pct = {ood_threshold:.3f}")
ax.set_title("Week 4: OOD proxy via nearest-neighbor distance")
ax.set_xlabel("distance to nearest RANDOM-context point")
ax.set_ylabel("count")
ax.legend()
plt.tight_layout()
out_path = FIGURES_DIR / "week4_ood_distance_hist.png"
plt.savefig(out_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved figure: {out_path}")
print(f"OOD rate (above 95th percentile threshold): {ood_rate:.4f}")


In [ ]:
bts_pred_chosen = model.predict_proba(
    np.hstack([
        context_bts,
        np.eye(N_ACTIONS)[feedback_bts["action"]]
    ])
)[:, 1]

risk_df = pd.DataFrame(
    {
        "pred_reward": bts_pred_chosen[sample_bts_idx],
        "nn_distance": distances,
    }
)

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    risk_df["nn_distance"],
    risk_df["pred_reward"],
    alpha=0.25,
    s=10,
    c=risk_df["pred_reward"],
    cmap="viridis",
)
ax.axvline(ood_threshold, color="crimson", linestyle="--", linewidth=2, label="OOD threshold (95th pct)")
ax.set_title("Week 4: DM risk map (confidence vs OOD)")
ax.set_xlabel("OOD proxy (NN distance)")
ax.set_ylabel("Predicted reward for logged action")
ax.legend(loc="upper right")
plt.colorbar(sc, ax=ax, label="predicted reward")
plt.tight_layout()
out_path = FIGURES_DIR / "week4_dm_risk_map.png"
plt.savefig(out_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved figure: {out_path}")
